- MUDANÇA NOS TIPOS DE MODELOS PARA SEREM MULTILINGUAGEM E LIDERAEM MELHOR COM PROBLEMAS SIMPLES DE SIMILARIDADE QUE DEIXAVA PASSAR APENAS PELA BARREIRA DO IDIOMA

threshold=0.5

self.bi_model = SentenceTransformer('intfloat/multilingual-e5-base').to(self.device) #paraphrase-multilingual-mpnet-base-v2

self.cross_model = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', device=self.device) #cross-encoder/ms-marco-MiniLM-L-6-v2


In [1]:
import pandas as pd
import io

file_path = '../../data/skill_taxonomy.csv'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    processed_lines = []
    for line in lines:
        parts = line.strip().split(',', 2) 
        if len(parts) == 3:
            processed_lines.append(parts)
    
    df_tax = pd.DataFrame(processed_lines[1:], columns=processed_lines[0])

    categorias = df_tax['skill_name'].unique()

    print(f"Total de categorias carregadas: {len(categorias)}")
    print("-" * 30)
    print("Categorias disponíveis:")
    for cat in sorted(categorias):
        print(f"- {cat}")

except FileNotFoundError:
    print("Arquivo não encontrado. Verifique o caminho.")
except Exception as e:
    print(f"Ocorreu um erro ao processar o arquivo: {e}")

Total de categorias carregadas: 100
------------------------------
Categorias disponíveis:
- AWS
- Adaptabilidade
- Análise Financeira
- Análise de Dados
- Aprendizado Contínuo
- Apresentação e Oratória
- Arquitetura de Software
- Autogestão
- Avaliação de Modelos
- Azure
- Bash e Shell Script
- CI/CD
- Colaboração
- Compliance e Regulação
- Comunicação Assertiva
- Cultura Organizacional
- Data Warehouse
- Deep Learning
- Desenvolvimento Organizacional
- Desenvolvimento de APIs
- Design Thinking
- Docker
- ETL e ELT
- Embeddings e Busca Vetorial
- Employer Branding
- Empreendedorismo
- Engenharia de Dados
- Engenharia de Features
- Escrita Profissional
- Estatística Aplicada
- Excel Avançado
- Experimentação e Testes A/B
- Facilitação
- FastAPI
- Foco no Cliente
- Gestão de Desempenho
- Gestão de Mudanças
- Gestão de Pessoas
- Gestão de Produto
- Gestão de Projetos
- Gestão de Riscos
- Gestão de Stakeholders
- Gestão do Tempo
- Google Cloud Platform
- Hadoop
- Hugging Face
- Infrastruc

- avaliacao

mapeamento_final_bi_cross.csv: É a visão geral de tudo o que foi processado. Contém as skills que deram match e as que não deram. É aqui que calculamos a saúde global. 🌍

auditoria_necessaria.csv: É um "filtro de inteligência". Ele contém apenas os casos onde a IA (ai_rerank) teve que agir. É o melhor lugar para encontrar erros de processo (como falhas de tradução) ou ambiguidades.

In [2]:
import pandas as pd

df_final = pd.read_csv('mapeamento_final_bi_cross.csv')
df_audit = pd.read_csv('auditoria_necessaria.csv')

total = len(df_final)
matches = df_final[df_final['match_status'] == 'matched']
no_matches = df_final[df_final['match_status'] == 'no_match']

print(f"Total de registros: {total}")
print(f"Matches confirmados: {len(matches)} ({len(matches)/total:.1%})")
print(f"No Matches (Gaps): {len(no_matches)} ({len(no_matches)/total:.1%})")

Total de registros: 3030
Matches confirmados: 1811 (59.8%)
No Matches (Gaps): 1219 (40.2%)


In [3]:
df_final.head(45)

,new_skill_id,skill_raw,taxonomy_id,skill_name,confidence_score,match_status,strategy_used
0,1,gestão de pessoas,1,Gestão de Pessoas,1.00,matched,fuzzy_match
1,2,People Management,1,Gestão de Pessoas,1.00,matched,ai_rerank
2,3,SQL,32,SQL,1.00,matched,fuzzy_match
3,4,sql avançado,58,PostgreSQL,0.82,matched,ai_rerank
4,5,machine learning,40,Machine Learning,1.00,matched,fuzzy_match
5,6,comunicação assertiva,2,Comunicação Assertiva,1.00,matched,fuzzy_match
6,7,python,31,Python,1.00,matched,fuzzy_match
7,8,PYTHON,31,Python,1.00,matched,fuzzy_match
8,9,análise de dados,39,Análise de Dados,1.00,matched,fuzzy_match
9,10,resolução de conflitos,3,Resolução de Conflitos,1.00,matched,fuzzy_match


In [4]:
df_audit.head(35)

,new_skill_id,skill_raw,taxonomy_id,skill_name,confidence_score,match_status,strategy_used
0,1018,consistência de dados,57,Qualidade de Dados,0.50,matched,ai_rerank
1,1980,KPI,91,Análise Financeira,0.50,matched,ai_rerank
2,2983,consistência de dados,57,Qualidade de Dados,0.50,matched,ai_rerank
3,2255,Excel,78,Excel Avançado,0.51,matched,ai_rerank
4,2816,raciocínio crítico,6,Pensamento Crítico,0.51,matched,ai_rerank
5,2613,excel,78,Excel Avançado,0.51,matched,ai_rerank
6,591,PBI,91,Análise Financeira,0.51,matched,ai_rerank
7,950,PBI,91,Análise Financeira,0.51,matched,ai_rerank
8,1157,deploy com docker,63,Docker,0.51,matched,ai_rerank
9,792,Excel,78,Excel Avançado,0.51,matched,ai_rerank


Respondendo "Por que não deu Match?"

Para saber se o problema é a Taxonomia (falta a skill) ou o Processo (a IA não viu a tradução),  olhar os no_matches.

Se uma skill aparece muito como no_match, ela é um forte candidato para entrar na sua taxonomia oficial.

In [5]:
#ordenando as linhas na ordem nos confidence mais altos para menores
df_final[df_final["match_status"] == "no_match"].sort_values('confidence_score', ascending=False).head(35)

,new_skill_id,skill_raw,taxonomy_id,skill_name,confidence_score,match_status,strategy_used
2863,2864,code review,—,—,0.50,no_match,insufficient_confidence
2137,2138,code review,—,—,0.50,no_match,insufficient_confidence
790,791,code review,—,—,0.50,no_match,insufficient_confidence
1614,1615,guiding others,—,—,0.49,no_match,insufficient_confidence
2208,2209,IaC,—,—,0.49,no_match,insufficient_confidence
2401,2402,análise de causa raiz,—,—,0.49,no_match,insufficient_confidence
1626,1627,análise de causa raiz,—,—,0.49,no_match,insufficient_confidence
564,565,contratação,—,—,0.49,no_match,insufficient_confidence
1062,1063,LLM deployment,—,—,0.49,no_match,insufficient_confidence
278,279,gestão de conflitos,—,—,0.49,no_match,insufficient_confidence


In [6]:
#analisando os 10 maiores Gaps
print("Top 10 skills que precisam de atenção (Gaps):")
no_matches['skill_raw'].value_counts().head(50)


Top 10 skills que precisam de atenção (Gaps):


skill_raw
comunicação com stakeholders           6
soluções inovadoras                    6
Git                                    5
visualização de dados                  5
integridade de dados                   5
estratégia de produto                  5
normalização de dados                  5
intrapreneurship                       5
DW                                     5
organizational design                  5
gestão de capital humano               4
versatilidade                          4
js                                     4
apache airflow                         4
pull request                           4
detecção de anomalias                  4
design técnico                         4
REST API                               4
context switching                      4
trabalhar sem supervisão               4
indicadores-chave                      4
data integrity                         4
PSQL                                   4
redshift                               4
Terraf

Respondendo "O Match está correto?"

usar o arquivo de auditoria para ver onde a IA está sofrendo para decidir. Scores baixos indicam que o match pode estar na categoria errada.

In [7]:
#ordenando pelos matches mais duvidosos (menor confiança)
print("Matches com menor confiança (potenciais erros de categoria):")
df_audit.sort_values('confidence_score').head(40)[['skill_raw', 'skill_name', 'confidence_score']]

Matches com menor confiança (potenciais erros de categoria):


,skill_raw,skill_name,confidence_score
0,consistência de dados,Qualidade de Dados,0.50
1,KPI,Análise Financeira,0.50
2,consistência de dados,Qualidade de Dados,0.50
3,Excel,Excel Avançado,0.51
4,raciocínio crítico,Pensamento Crítico,0.51
5,excel,Excel Avançado,0.51
6,PBI,Análise Financeira,0.51
7,PBI,Análise Financeira,0.51
8,deploy com docker,Docker,0.51
9,Excel,Excel Avançado,0.51
